## Write Urself Challenge
this is a section where I have to rewrite everything from memory and figure it out myself

In [61]:
import math
import random

In [62]:
class Value:
    def __init__(self, num, _child=(), label=''):
        self.data   = num
        self.label  = label

        self._prev  = set(_child)
        self._backward = lambda: None
        self.grad   = 0.0

    # print
    def __repr__(self):
        return f"Value({self.label}:{self.data})"


    # basic operations
    def __add__ (self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad 
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad  += other.data * out.grad
            other.grad +=  self.data * out.grad 
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += other * (self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1 

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def exp(self):
        out = Value(math.exp(self.data), (self,))

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    # activation func
    def sigmoid(self):
        out = Value(1 / (1 + (-self).exp().data), (self,))

        def _backward():
            self.grad += out.data * (1 - out.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self,))

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        t = max(0, self.data)
        out = Value(t, (self,))
        
        def _backward():
            self.grad += (1 if t > 0 else 0) * out.grad
        out._backward = _backward

        return out

    # backprop
    def backward(self):
        topo=[]
        vis =set()

        def build_topo(val):
            if val not in vis:
                vis.add(val)
                for c in val._prev:     
                    build_topo(c)
                topo.append(val)

        build_topo(self)
        self.grad = 1.0
        for val in reversed(topo):
            val._backward()

In [59]:
a = Value(4.0, label='a');
b = Value(3.0, label='b')
c = Value(1.0, label='c')

d = a * b; d.label='d'
e = d + c; e.label='e'
s = e.relu()

s.backward()
s


Value(:13.0)

In [60]:
print(s.grad)
print(e.grad)
print(d.grad)
print(c.grad)
print(b.grad)
print(a.grad)

1.0
1.0
1.0
1.0
4.0
3.0


### NEURAL NETWORK I AM COMING BABYYYY

In [ ]:
class Neuron:
    def __init__(self, nin, activation=None):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b =  Value(random.uniform(-1,1))

        acts = {
            'relu': lambda x: x.relu(),
            'sigmoid': lambda x: x.sigmoid(),
            'tanh': lambda x: x.tanh(),
            'identity': lambda x: x,
        }
        self.activation = acts[activation]

    def __call__(self, x):
        out = sum((xi*wi for xi, wi in zip(x, self.w)), self.b)
        return self.activation(out)

    def parameters(self):
        return [self.b] + self.w

In [97]:
class Layer:
    def __init__(self, nin, nout, activation=None):
        self.neurons = [Neuron(nin, activation=activation) for _ in range(nout)] 

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

In [ ]:
x = [2.0, 3.0, 4.0]


output: 0.8768236288524188
weights grads:
0 0.46236064777215136
1 0.693540971658227
2 0.9247212955443027


[Value(:0.5770663213447602),
 Value(:-0.6826754957451404),
 Value(:-0.5706604874228396),
 Value(:0.9655313865680875)]